# **Data Preprocessing & Feature Engineering**
## AI-Driven Coronary Disease Detection and Decision Support System
### Patient Recommendation System

In [2]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive to access files
from google.colab import drive
drive.mount('/content/drive')

# Load the dataset
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DSGP/Heart_health new.csv")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")


Mounted at /content/drive
Dataset shape: (724, 12)
Columns: ['ID', 'Name', 'Age', 'Gender', 'Height cm', 'Weight kg', 'Blood Pressure mmHg', 'Cholesterol mg/dL', 'Glucose mg/dL', 'Smoker', 'Exercise hours/week', 'Heart Attack']



# **1. Data Cleaning**
This section deals with cleaning the dataset by handling missing values, duplicates, and ensuring data types are correct.


In [4]:
# ======================
# 1. DATA CLEANING
# ======================
print("\n" + "="*50)
print("1. DATA CLEANING")
print("="*50)

# Check for missing values
print("Missing values check:")
print(df.isnull().sum())

# Check for duplicates
duplicates = df.duplicated().sum()
print(f"\nDuplicate rows: {duplicates}")
if duplicates > 0:
    df = df.drop_duplicates()
    print(f"Removed {duplicates} duplicates")

# Check data types
print("\nData types:")
print(df.dtypes)



1. DATA CLEANING
Missing values check:
ID                     0
Name                   0
Age                    0
Gender                 0
Height cm              0
Weight kg              0
Blood Pressure mmHg    0
Cholesterol mg/dL      0
Glucose mg/dL          0
Smoker                 0
Exercise hours/week    0
Heart Attack           0
dtype: int64

Duplicate rows: 10
Removed 10 duplicates

Data types:
ID                      int64
Name                   object
Age                     int64
Gender                 object
Height cm               int64
Weight kg               int64
Blood Pressure mmHg    object
Cholesterol mg/dL       int64
Glucose mg/dL           int64
Smoker                 object
Exercise hours/week     int64
Heart Attack            int64
dtype: object



# **2. Basic Feature Engineering**
This step focuses on basic feature engineering, such as splitting blood pressure into systolic and diastolic, calculating BMI, and creating categories for age, cholesterol, glucose, and blood pressure.


In [6]:

# ======================
# 2. BASIC FEATURE ENGINEERING
# ======================
print("\n" + "="*50)
print("2. FEATURE ENGINEERING")
print("="*50)

# 2.1 Split Blood Pressure into two columns
df[['Systolic_BP', 'Diastolic_BP']] = df['Blood Pressure mmHg'].str.split('/', expand=True).astype(int)
print("✓ Split Blood Pressure into Systolic and Diastolic")

# 2.2 Calculate BMI (Body Mass Index)
# BMI = weight(kg) / height(m)^2
df['Height_m'] = df['Height cm'] / 100  # Convert cm to meters
df['BMI'] = df['Weight kg'] / (df['Height_m'] ** 2)
print("✓ Calculated BMI")

# 2.3 Create Age Groups
df['Age_Group'] = pd.cut(df['Age'],
                        bins=[20, 35, 50, 65, 100],
                        labels=['Young', 'Middle', 'Senior', 'Elderly'])
print("✓ Created Age Groups")

# 2.4 Create Cholesterol Categories
df['Cholesterol_Level'] = pd.cut(df['Cholesterol mg/dL'],
                                bins=[0, 200, 240, 1000],
                                labels=['Normal', 'Borderline', 'High'])
print("✓ Created Cholesterol Categories")

# 2.5 Create Glucose Categories
df['Glucose_Level'] = pd.cut(df['Glucose mg/dL'],
                           bins=[0, 100, 126, 1000],
                           labels=['Normal', 'Prediabetic', 'Diabetic'])
print("✓ Created Glucose Categories")

# 2.6 Create BP Categories
def categorize_bp(systolic, diastolic):
    if systolic < 120 and diastolic < 80:
        return 'Normal'
    elif systolic < 130 and diastolic < 80:
        return 'Elevated'
    elif systolic < 140 or diastolic < 90:
        return 'Stage1_Hypertension'
    else:
        return 'Stage2_Hypertension'

df['BP_Category'] = df.apply(lambda x: categorize_bp(x['Systolic_BP'], x['Diastolic_BP']), axis=1)
print("✓ Created BP Categories")

# 2.7 Create Risk Flags
# High cholesterol flag
df['High_Cholesterol'] = (df['Cholesterol mg/dL'] > 200).astype(int)

# High glucose flag
df['High_Glucose'] = (df['Glucose mg/dL'] > 100).astype(int)

# Hypertension flag
df['Hypertension'] = ((df['Systolic_BP'] >= 140) | (df['Diastolic_BP'] >= 90)).astype(int)

# Obesity flag (BMI >= 30)
df['Obesity'] = (df['BMI'] >= 30).astype(int)

print("✓ Created Risk Flags")

# 2.8 Create Composite Risk Score (Simple version)
def calculate_risk_score(row):
    score = 0
    if row['Age'] > 50:
        score += 1
    if row['Smoker'] == 'Yes':
        score += 1
    if row['High_Cholesterol'] == 1:
        score += 1
    if row['Hypertension'] == 1:
        score += 1
    if row['Obesity'] == 1:
        score += 1
    if row['Exercise hours/week'] < 2:
        score += 1
    return score

df['Risk_Score'] = df.apply(calculate_risk_score, axis=1)
df['Risk_Level'] = pd.cut(df['Risk_Score'],
                         bins=[-1, 2, 4, 7],
                         labels=['Low', 'Medium', 'High'])
print("✓ Calculated Risk Score and Level")



2. FEATURE ENGINEERING
✓ Split Blood Pressure into Systolic and Diastolic
✓ Calculated BMI
✓ Created Age Groups
✓ Created Cholesterol Categories
✓ Created Glucose Categories
✓ Created BP Categories
✓ Created Risk Flags
✓ Calculated Risk Score and Level



# **3. Handling Categorical Data**
This section encodes categorical variables such as 'Smoker' and 'Gender'. We also perform one-hot encoding for categorical columns with more than two unique values, such as `Age_Group` and `BP_Category`.


In [8]:

# ======================
# 3. HANDLING CATEGORICAL DATA
# ======================
print("\n" + "="*50)
print("3. ENCODING CATEGORICAL VARIABLES")
print("="*50)

# 3.1 Binary encoding for Yes/No columns
df['Smoker_Encoded'] = df['Smoker'].map({'Yes': 1, 'No': 0})
print("✓ Encoded Smoker (Yes=1, No=0)")

# 3.2 Label encoding for Gender
df['Gender_Encoded'] = df['Gender'].map({'Male': 1, 'Female': 0})
print("✓ Encoded Gender (Male=1, Female=0)")

# 3.3 One-hot encoding for multi-category columns
categorical_cols = ['Age_Group', 'Cholesterol_Level', 'Glucose_Level', 'BP_Category']
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
print(f"✓ One-hot encoded: {categorical_cols}")



3. ENCODING CATEGORICAL VARIABLES
✓ Encoded Smoker (Yes=1, No=0)
✓ Encoded Gender (Male=1, Female=0)
✓ One-hot encoded: ['Age_Group', 'Cholesterol_Level', 'Glucose_Level', 'BP_Category']



# **4. Handling Numerical Data**
In this section, we apply transformations such as log transformations for skewed data and standard scaling to the numerical columns.


In [13]:

# ======================
# 4. HANDLING NUMERICAL DATA
# ======================
print("\n" + "="*50)
print("4. SCALING NUMERICAL FEATURES")
print("="*50)

# Select numerical columns to scale
# Exclude ID, Name, and previously encoded/categorized columns, and target related columns
# Also exclude columns that will be dropped later (e.g., Blood Pressure mmHg, Height cm, Weight kg, Height_m)

# Get all numeric columns that are not already handled or are not target/identifier columns

# Columns that are definitely numerical and need scaling/transformation
numerical_cols_to_process = [
    'Age', 'Height cm', 'Weight kg', 'Cholesterol mg/dL',
    'Glucose mg/dL', 'Exercise hours/week', 'Systolic_BP',
    'Diastolic_BP', 'BMI', 'Risk_Score'
]

# Filter to only include columns that exist in df_encoded
actual_numerical_cols = [col for col in numerical_cols_to_process if col in df_encoded.columns]

print(f"\nNumerical columns identified for processing: {actual_numerical_cols}")

# Check for outliers using IQR (optional, for insights)
print("\nOutlier detection (IQR method):")
for col in actual_numerical_cols:
    Q1 = df_encoded[col].quantile(0.25)
    Q3 = df_encoded[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df_encoded[(df_encoded[col] < lower_bound) | (df_encoded[col] > upper_bound)]
    if len(outliers) > 0:
        print(f"  {col}: {len(outliers)} outliers detected ({len(outliers)/len(df_encoded)*100:.1f}%) -- Consider Winsorization or removal if severe.")

# Apply log transformation to skewed features if needed
skewed_cols = ['Cholesterol mg/dL', 'Glucose mg/dL', 'Exercise hours/week'] # Assuming these might be skewed
for col in skewed_cols:
    if col in df_encoded.columns:
        # Add small constant to avoid log(0) for features that can be 0
        if (df_encoded[col] >= 0).all(): # Check if all values are non-negative
            df_encoded[f'{col}_log'] = np.log1p(df_encoded[col])
            print(f"✓ Applied log transformation to {col}")
        else:
            print(f"Skipped log transformation for {col} due to negative values.")

# Update numerical_cols for scaling to include log-transformed ones and exclude original if log-transformed
scaled_cols = []
for col in actual_numerical_cols:
    if col in skewed_cols and f'{col}_log' in df_encoded.columns:
        scaled_cols.append(f'{col}_log')
    else:
        scaled_cols.append(col)

# Standard scaling
scaler = StandardScaler()
print("\nApplying Standard Scaling...")
for col in scaled_cols:
    if col in df_encoded.columns:
        df_encoded[f'{col}_scaled'] = scaler.fit_transform(df_encoded[[col]])
        print(f"  Scaled: {col}")

print("✓ Numerical features processed and scaled.")



4. SCALING NUMERICAL FEATURES

Numerical columns identified for processing: ['Age', 'Height cm', 'Weight kg', 'Cholesterol mg/dL', 'Glucose mg/dL', 'Exercise hours/week', 'Systolic_BP', 'Diastolic_BP', 'BMI', 'Risk_Score']

Outlier detection (IQR method):
  Glucose mg/dL: 5 outliers detected (0.7%) -- Consider Winsorization or removal if severe.
  Systolic_BP: 7 outliers detected (1.0%) -- Consider Winsorization or removal if severe.
  Diastolic_BP: 1 outliers detected (0.1%) -- Consider Winsorization or removal if severe.
  BMI: 7 outliers detected (1.0%) -- Consider Winsorization or removal if severe.
✓ Applied log transformation to Cholesterol mg/dL
✓ Applied log transformation to Glucose mg/dL
✓ Applied log transformation to Exercise hours/week

Applying Standard Scaling...
  Scaled: Age
  Scaled: Height cm
  Scaled: Weight kg
  Scaled: Cholesterol mg/dL_log
  Scaled: Glucose mg/dL_log
  Scaled: Exercise hours/week_log
  Scaled: Systolic_BP
  Scaled: Diastolic_BP
  Scaled: BMI
  S


# **5. Integrating Group Outputs (Mock Data)**
In this section, we simulate data from other group members (e.g., blockage percentage, engagement probability) and integrate these outputs into the dataset.


In [15]:

# ======================
# 5. INTEGRATING GROUP OUTPUTS (MOCK)
# ======================
print("\n" + "="*50)
print("5. INTEGRATING GROUPMATE OUTPUTS (MOCK DATA)")
print("="*50)

# Mock data from other group members
def mock_group_outputs():
    np.random.seed(42)
    n_samples = len(df)

    # Mock blockage data (from Desindu)
    blockage_percentage = np.random.uniform(0, 90, n_samples)
    arteries_affected = np.random.choice([0, 1, 2, 3], n_samples, p=[0.3, 0.4, 0.2, 0.1])

    # Mock risk prediction (from Mevin)
    engagement_prob = np.random.uniform(0.3, 0.95, n_samples)

    # Mock ECG metadata (from Chamath)
    ecg_anomaly = np.random.choice([0, 1], n_samples, p=[0.7, 0.3])

    return {
        'Blockage_Percentage': blockage_percentage,
        'Arteries_Affected': arteries_affected,
        'Engagement_Probability': engagement_prob,
        'ECG_Anomaly': ecg_anomaly
    }

# Add mock group outputs
group_outputs = mock_group_outputs()
for key, value in group_outputs.items():
    df_encoded[key] = value

# Create derived features from group outputs
df_encoded['Blockage_Severity'] = pd.cut(df_encoded['Blockage_Percentage'],
                                       bins=[0, 30, 60, 100],
                                       labels=['Mild', 'Moderate', 'Severe'])

df_encoded['Multiple_Arteries'] = (df_encoded['Arteries_Affected'] > 1).astype(int)

print("✓ Added mock group outputs:")
for key in group_outputs.keys():
    print(f"  - {key}")



5. INTEGRATING GROUPMATE OUTPUTS (MOCK DATA)
✓ Added mock group outputs:
  - Blockage_Percentage
  - Arteries_Affected
  - Engagement_Probability
  - ECG_Anomaly



# **6. Final Feature Selection**
Here, we finalize the features to be used in the recommendation system and save the processed data for future modeling.


In [17]:

# ======================
# 6. FINAL FEATURE SET FOR RECOMMENDATION SYSTEM
# ======================
print("\n" + "="*50)
print("6. FINAL FEATURE SET FOR RECOMMENDATION SYSTEM")
print("="*50)

# Select features for the model
final_features = [
    # Demographic
    'Age', 'Gender_Encoded', 'BMI',

    # Clinical Measurements
    'Systolic_BP', 'Diastolic_BP',
    'Cholesterol mg/dL', 'Glucose mg/dL',

    # Lifestyle
    'Smoker_Encoded', 'Exercise hours/week',

    # Risk Flags
    'High_Cholesterol', 'High_Glucose',
    'Hypertension', 'Obesity',

    # Risk Score
    'Risk_Score',

    # From groupmates
    'Blockage_Percentage', 'Arteries_Affected',
    'Engagement_Probability', 'ECG_Anomaly',
    'Multiple_Arteries',

    # Engineered categories (one-hot encoded)
    'Age_Group_Middle', 'Age_Group_Senior', 'Age_Group_Elderly',
    'Cholesterol_Level_High', 'Cholesterol_Level_Normal',
    'Glucose_Level_Diabetic', 'Glucose_Level_Normal',
    'BP_Category_Stage1_Hypertension', 'BP_Category_Stage2_Hypertension'
]

# Filter only features that exist in dataframe
available_features = [f for f in final_features if f in df_encoded.columns]
print(f"Total features available: {len(available_features)}")
print(f"Sample features: {available_features[:10]}...")

# Create final dataset
X = df_encoded[available_features]
y = df['Heart Attack']  # Target variable

print(f"\nFinal dataset shape: X={X.shape}, y={y.shape}")



6. FINAL FEATURE SET FOR RECOMMENDATION SYSTEM
Total features available: 25
Sample features: ['Age', 'Gender_Encoded', 'BMI', 'Systolic_BP', 'Diastolic_BP', 'Cholesterol mg/dL', 'Glucose mg/dL', 'Smoker_Encoded', 'Exercise hours/week', 'High_Cholesterol']...

Final dataset shape: X=(714, 25), y=(714,)



# **7. Save Processed Data**
In this final section, the processed features and target variable are saved into CSV files for future use.


In [19]:

# ======================
# 7. SAVE PROCESSED DATA
# ======================
# Save processed features
X.to_csv('processed_features.csv', index=False)
y.to_csv('target_variable.csv', index=False)

# Save complete processed dataset
processed_df = pd.concat([X, y], axis=1)
processed_df.to_csv('heart_disease_processed_complete.csv', index=False)

print("\n" + "="*50)
print("PREPROCESSING COMPLETE!")
print("="*50)
print(f"\nSaved files:")
print("1. processed_features.csv - Features for ML model")
print("2. target_variable.csv - Target labels")
print("3. heart_disease_processed_complete.csv - Complete processed dataset")
print(f"\nOriginal features: {len(df.columns)}")
print(f"Engineered features: {len(X.columns)}")
print(f"Total transformations: {len(df_encoded.columns) - len(df.columns)}")

# Display sample of processed data
print("\nSample of processed data (first 5 rows):")
print(processed_df[['Age', 'BMI', 'Risk_Score', 'Blockage_Percentage', 'Heart Attack']].head())



PREPROCESSING COMPLETE!

Saved files:
1. processed_features.csv - Features for ML model
2. target_variable.csv - Target labels
3. heart_disease_processed_complete.csv - Complete processed dataset

Original features: 28
Engineered features: 25
Total transformations: 24

Sample of processed data (first 5 rows):
   Age        BMI  Risk_Score  Blockage_Percentage  Heart Attack
0   35  25.390625           0            33.708611             0
1   40  25.711662           0            85.564288             0
2   30  24.973985           1            65.879455             0
3   38  25.910684           0            53.879264             0
4   42  25.510204           1            14.041678             0
